In [1]:
import evidently
import pandas as pd
import numpy as np
import requests
import zipfile
import io

from datetime import datetime, time
from sklearn import ensemble
from evidently import Report, Dataset, DataDefinition, Regression
from evidently.presets import DataDriftPreset, RegressionPreset

# download and unzip dataset
content = requests.get(
    "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
    verify=False
).content

with zipfile.ZipFile(io.BytesIO(content)) as arc:
    raw_data = pd.read_csv(arc.open("hour.csv"),
                           header=0,
                           sep=",",
                           parse_dates=["dteday"],
                           index_col="dteday")
raw_data.index = raw_data.apply(
    lambda row: datetime.combine(row.name, time(hour=int(row["hr"]))),
    axis=1
)

# create Evidently data definition to identify different feature types and target
target = "cnt"
prediction = "prediction"
numerical_features = ["temp", "atemp", "hum", "windspeed", "hr", "weekday"]
categorical_features = ["season", "holiday", "workingday"]

data_definition = DataDefinition(
    numerical_columns=numerical_features,
    categorical_columns=categorical_features,
    regression=[Regression(target=target,
                           prediction=prediction)]
)

# split data
reference = raw_data.loc[ : "2011-01-28 23:00:00"].copy()
current = raw_data.loc["2011-01-29 00:00:00" : ].copy()

# train regressor with reference data
regressor = ensemble.RandomForestRegressor(random_state=0, n_estimators=50)
regressor.fit(reference[numerical_features + categorical_features],
              reference[target])

# make predictions
reference[prediction] = regressor.predict(reference[numerical_features + categorical_features])
current[prediction] = regressor.predict(current[numerical_features + categorical_features])

reference_dataset = Dataset.from_pandas(reference, data_definition=data_definition)
current_dataset = Dataset.from_pandas(current, data_definition=data_definition)

# report on data drift and model
report = Report([
    DataDriftPreset(),
    RegressionPreset()
])
my_eval = report.run(current_dataset, reference_dataset)
my_eval

C:\Users\linho\anaconda3\envs\comp4948\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'archive.ics.uci.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


TypeError: 'DataFrame' object is not callable